# Phase 1: Data Collection and Structurally Faithful Preprocessing
This notebook pulls the raw financial datasets (FinQA for fine-tuning, FinanceBench for evaluation) and applies heuristic layout parsing. It converts flattened 2D arrays into structurally preserved Markdown tables and formats the training data to enforce Chain of Thought (CoT) reasoning.

In [ ]:
!pip install datasets pandas


In [16]:
import pandas as pd
import json
from datasets import load_dataset, Dataset
import os

# 1. Load the Datasets
print("Loading Datasets...")

# --- BULLETPROOF FINQA LOADING ---
# Load the raw JSON manually to avoid PyArrow mixed-type errors
finqa_path = "data/finQA/train.json"
with open(finqa_path, "r", encoding="utf-8") as f:
    raw_finqa_data = json.load(f)

# Extract ONLY the fields we need into a clean list
cleaned_finqa = []
for item in raw_finqa_data:
    qa_data = item.get("qa", {})
    
    cleaned_finqa.append({
        "pre_text": item.get("pre_text", []),
        "post_text": item.get("post_text", []),
        "table": item.get("table", []),
        "question": qa_data.get("question", ""),
        "answer": str(qa_data.get("answer", "")), # Force string to prevent type errors
        # Handle the fact that reasoning steps are sometimes called 'exe_list' or 'steps'
        "exe_list": qa_data.get("exe_list", qa_data.get("steps", [])) 
    })

# Convert the clean data into a Hugging Face Dataset
finqa_dataset = Dataset.from_list(cleaned_finqa)

# Load FinanceBench from HF Hub
financebench_dataset = load_dataset("PatronusAI/financebench", split="train")

print(f"FinQA samples: {len(finqa_dataset)} | FinanceBench samples: {len(financebench_dataset)}")

# 2. Define Structurally Faithful Parsing
def format_table_to_markdown(table_data):
    """Converts raw 2D array financial tables into structurally faithful Markdown."""
    if not table_data or len(table_data) == 0:
        return ""
    
    try:
        headers = table_data[0]
        rows = table_data[1:]
        df = pd.DataFrame(rows, columns=headers)
        return df.to_markdown(index=False)
    except Exception as e:
        return str(table_data)

# 3. Define Chain of Thought (CoT) Prompt Formatting
def format_cot_prompt(example):
    """Formats FinQA into a strict CoT prompt for SLM fine-tuning."""
    
    # We already flattened the dataset in step 1, so extraction is simple!
    pre_text = example['pre_text']
    post_text = example['post_text']
    table_data = example['table']
    question = example['question']
    answer = example['answer']
    exe_list = example['exe_list']

    # Ensure context variables are strings
    pre_text_str = " ".join(pre_text) if isinstance(pre_text, list) else str(pre_text)
    post_text_str = " ".join(post_text) if isinstance(post_text, list) else str(post_text)
    text_context = pre_text_str + "\n" + post_text_str
    
    table_context = format_table_to_markdown(table_data)
    
    # Handle exe_list which could be a list of strings or a list of dictionaries
    if isinstance(exe_list, list) and len(exe_list) > 0 and isinstance(exe_list[0], dict):
        # Extract the 'res' or math operations if it's a list of dicts
        reasoning_steps = " -> ".join([f"{step.get('op', '')}({step.get('arg1', '')}, {step.get('arg2', '')}) = {step.get('res', '')}" for step in exe_list])
    else:
        reasoning_steps = " ".join(exe_list) if isinstance(exe_list, list) else str(exe_list)
    
    instruction = "You are a highly accurate financial AI. Using the provided context, answer the question by thinking step-by-step."
    
    full_prompt = f"""[INST] {instruction}
Context:
{text_context}

Financial Table:
{table_context}

Question: {question}
[/INST]
Let's think step by step. 
Reasoning: {reasoning_steps}
Final Answer: {answer}"""
    
    return {"formatted_prompt": full_prompt}

# 4. Apply Formatting and Save
print("Applying CoT formatting and parsing tables...")
finqa_cot_dataset = finqa_dataset.map(format_cot_prompt)

# Save to disk
os.makedirs("processed_data", exist_ok=True)
finqa_cot_dataset.save_to_disk("processed_data/finqa_cot")
financebench_dataset.save_to_disk("processed_data/financebench")

print("Preprocessing complete. Data saved to processed_data/")

Loading Datasets...
FinQA samples: 6251 | FinanceBench samples: 150
Applying CoT formatting and parsing tables...


Saving the dataset (1/1 shards): 100%|██████████| 150/150 [00:00<00:00, 32608.35 examples/s]

Preprocessing complete. Data saved to processed_data/
